# 22 · Gray-Box AIA72 AIA-only fallback feasibility

Phase F asks whether the already-frozen AIA branch is sufficiently reliable to act as the degraded branch when SHARP is unavailable. This is a post-hoc feasibility audit only: no new fitting, recalibration, threshold selection, gating rule, or policy redesign.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

ROOT=Path('/home/abmoses2000')
MASTER=ROOT/'graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz'
SHARP=ROOT/'sharp72_predictions_frozen_20261002.csv.gz'
MOND=ROOT/'graybox_aia72_conformal_mondrian_v2_20261005/mondrian_thresholds.csv'
ALARM=ROOT/'graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005/alarm_thresholds.csv'
OUT=ROOT/'graybox_aia72_aia_only_fallback_feasibility_v1_20261006'
if OUT.exists(): raise SystemExit(f'Refusing to overwrite {OUT}')
OUT.mkdir(parents=True)

a=pd.read_csv(MASTER,low_memory=False)
s=pd.read_csv(SHARP,low_memory=False)
for d in (a,s):
    d['forecast_case_id']=d['forecast_case_id'].astype(str)
m=a.merge(s,on='forecast_case_id',suffixes=('_aia','_sharp'),validate='one_to_one')

la='label_aia' if 'label_aia' in m.columns else 'label'
ls='label_sharp' if 'label_sharp' in m.columns else la
m['label_final']=pd.to_numeric(m[la],errors='raise').astype(int)
if not np.array_equal(m.label_final.to_numpy(),pd.to_numeric(m[ls],errors='raise').astype(int).to_numpy()):
    raise ValueError('Label mismatch')

role_col='role_aia' if 'role_aia' in m.columns else ('role_sharp' if 'role_sharp' in m.columns else 'role')
m['role_final']=m[role_col].astype(str)
issue=pd.Series(pd.NaT,index=m.index,dtype='datetime64[ns, UTC]')
for c in [x for x in ['issue_utc_aia','issue_utc_sharp','issue_utc'] if x in m.columns]:
    z=pd.to_datetime(m[c],utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
id_time=m.forecast_case_id.str.extract(r':(\d{8}_\d{4})_',expand=False)
z=pd.to_datetime(id_time,format='%Y%m%d_%H%M',utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
if issue.isna().any(): raise ValueError('Unresolved issue times')
m['year']=issue.dt.year.astype(int)

m['sharp_probability']=pd.to_numeric(m['probability_gru_mean'],errors='coerce')
m['aia_probability']=pd.to_numeric(m['probability'],errors='coerce')
m['fusion_probability']=.5*(m.sharp_probability+m.aia_probability)
def regime(r):
    if r.role_final=='retrospective_cycle25': return 'cycle25_2021_2025_diagnostic'
    if r.role_final=='supplementary_2026': return 'supplementary_2026_locked'
    if r.year<=2019: return 'cycle24_earlier_development'
    return 'other_or_gap'
m['regime']=m.apply(regime,axis=1)

alarm=pd.read_csv(ALARM)
ath={r.branch:float(r.threshold) for r in alarm.itertuples()}
m['sharp_alarm']=(m.sharp_probability>=ath['sharp']).astype(int)
m['aia_alarm']=(m.aia_probability>=ath['aia']).astype(int)
m['fusion_alarm']=(m.fusion_probability>=ath['sharp_aia']).astype(int)

mond=pd.read_csv(MOND)
z=mond[(mond.branch=='aia')&np.isclose(mond.alpha.astype(float),.10)]
if len(z)!=1: raise ValueError('Missing AIA q90 Mondrian thresholds')
q0,q1=float(z.iloc[0].qhat_class0),float(z.iloc[0].qhat_class1)
p=m.aia_probability.to_numpy(float)
inc0=p<=q0; inc1=(1-p)<=q1
m['aia_set_singleton']=inc0 ^ inc1
m['aia_set_both']=inc0 & inc1
m['aia_set_empty']=~inc0 & ~inc1
y=m.label_final.to_numpy(int)
m['aia_contains_true']=np.where(y==1,inc1,inc0)

status='input_status' if 'input_status' in m.columns else ('input_status_aia' if 'input_status_aia' in m.columns else None)
m['aia_available']=np.isfinite(m.aia_probability)
if status is not None:
    m['aia_available'] &= m[status].astype(str).eq('ok')

def metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=((y==1)&(pred==1)).sum(); fp=((y==0)&(pred==1)).sum()
    fn=((y==1)&(pred==0)).sum(); tn=((y==0)&(pred==0)).sum()
    rec=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    return dict(tp=int(tp),fp=int(fp),fn=int(fn),tn=int(tn),recall=rec,specificity=spec,tss=rec+spec-1)

rows=[]
for rg,g in m.groupby('regime',sort=False):
    yy=g.label_final.to_numpy(int)
    for system,pcol,acol in [('sharp','sharp_probability','sharp_alarm'),('aia_only','aia_probability','aia_alarm'),('fusion','fusion_probability','fusion_alarm')]:
        prob=g[pcol].to_numpy(float)
        issued=np.isfinite(prob)
        if system=='aia_only': issued &= g.aia_available.to_numpy(bool)
        y2=yy[issued]; p2=prob[issued]; pred=g.loc[issued,acol].to_numpy(int)
        r={'regime':rg,'system':system,'cases':len(g),'issued_cases':int(issued.sum()),'availability_rate':float(issued.mean()),
           'positive_retention_rate':float(issued[yy==1].mean()) if (yy==1).any() else np.nan}
        if len(y2):
            r.update(metrics(y2,pred))
            r['brier']=float(brier_score_loss(y2,p2))
            r['average_precision']=float(average_precision_score(y2,p2)) if len(np.unique(y2))==2 else np.nan
            r['roc_auc']=float(roc_auc_score(y2,p2)) if len(np.unique(y2))==2 else np.nan
        rows.append(r)
summary=pd.DataFrame(rows)
summary.to_csv(OUT/'branch_feasibility_summary.csv',index=False)

conf=[]
for rg,g in m.groupby('regime',sort=False):
    flare=g.label_final.eq(1) & g.aia_available
    conf.append({'regime':rg,'cases':len(g),'positives':int(g.label_final.eq(1).sum()),
                 'aia_available_rate':float(g.aia_available.mean()),
                 'aia_q90_singleton_rate':float(g.loc[g.aia_available,'aia_set_singleton'].mean()),
                 'aia_q90_both_rate':float(g.loc[g.aia_available,'aia_set_both'].mean()),
                 'aia_q90_flare_coverage':float(g.loc[flare,'aia_contains_true'].mean()) if flare.any() else np.nan,
                 'aia_alarm_recall':float(g.loc[flare,'aia_alarm'].mean()) if flare.any() else np.nan})
pd.DataFrame(conf).to_csv(OUT/'aia_uncertainty_feasibility.csv',index=False)

protocol={'status':'POSTHOC_AIA_ONLY_FALLBACK_FEASIBILITY_COMPLETE','created_utc':datetime.now(timezone.utc).isoformat(),
          'horizon_hours':72,'new_model_fitting':False,'new_threshold_tuning':False,'new_policy_selection':False,'new_policy_redesign':False,
          'question':'Can the frozen AIA branch support degraded issuance when SHARP is unavailable?',
          'decision_rule':'No new pass/fail threshold is introduced. Interpret branch skill, recall and flare-class conformal transfer jointly.',
          'evidence_boundary':{'cycle24':'development/descriptive','cycle25':'diagnostic/post-hoc','2026':'already-spent locked evidence'},
          'outputs':['branch_feasibility_summary.csv','aia_uncertainty_feasibility.csv']}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')

print('===== AIA-ONLY FALLBACK FEASIBILITY COMPLETE =====')
print(summary.round(6).to_string(index=False))
print(pd.DataFrame(conf).round(6).to_string(index=False))
print('Outputs:',OUT)


===== AIA-ONLY FALLBACK FEASIBILITY COMPLETE =====
                      regime   system  cases  issued_cases  availability_rate  positive_retention_rate   tp    fp   fn    tn   recall  specificity      tss    brier  average_precision  roc_auc
 cycle24_earlier_development    sharp  24046         24046           1.000000                 1.000000 1127  2545  279 20095 0.801565     0.887588 0.689153 0.036542           0.538052 0.921556
 cycle24_earlier_development aia_only  24046         20141           0.837603                 0.704125  902 10888   88  8263 0.911111     0.431466 0.342577 0.046872           0.121891 0.768481
 cycle24_earlier_development   fusion  24046         24046           1.000000                 1.000000 1165  4488  241 18152 0.828592     0.801767 0.630359 0.041436           0.524487 0.899188
cycle25_2021_2025_diagnostic    sharp  35846         35846           1.000000                 1.000000 2811  5992 1049 25994 0.728238     0.812668 0.540906 0.077365           0.

## Interpretation guardrails

A negative result is useful: it may justify preserving ABSTAIN when SHARP is unavailable rather than forcing an AIA-only forecast. Any future AIA-only fallback policy would need earlier-data development and separate prospective validation.
